<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/Fine_Tuning_LLMs_QLoRA_Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning LLMs — Hands-on Lab
## Teach a small model to answer support tickets in *your* house style (QLoRA + SFT)

**Companion to:** *Fine-Tuning LLMs, Part 1 & Part 2*

---

### What you will build

A tiny open model (**Qwen2.5-0.5B-Instruct**, Apache-2.0, no login or access request needed) that replies to
customer-support messages in one fixed, scannable format — every single time.

You will see the model **before** and **after** training on the same held-out questions. That visible change
is the whole point of the lab.

### What it costs

| | |
|---|---|
| **GPU** | Free Google Colab **T4** |
| **Paid account** | Not needed |
| **API key / HF token** | Not needed — the model is ungated |
| **Training time** | ~5–10 minutes |
| **Adapter you keep** | a few MB |

### How to run it

1. Open this notebook in **Google Colab**.
2. `Runtime → Change runtime type → T4 GPU` → Save.
3. `Runtime → Run all`, or run the cells one at a time from the top.
4. After the install cell, Colab may ask you to **restart the session**. Restart, then continue from the *next* cell.

### If you don't write code

You can run every cell top-to-bottom without editing anything. Your job is to **read the printed outputs** and
the comments. Two cells are marked **"TRY EDITING THIS"** — changing those and re-running is where the real
intuition comes from.

---

### The five steps

| # | Step | Concept from the deck |
|---|---|---|
| 1 | Build a small, perfectly consistent dataset | *Data quality decides the outcome* |
| 2 | Look at the base model **before** training | *Freeze a baseline first* |
| 3 | Load the model in 4-bit and attach LoRA | *QLoRA + PEFT* |
| 4 | Train with `SFTTrainer` | *SFT* |
| 5 | Compare, then save the adapter | *The adapter is a file you own* |

---
## Step 0 — Install the libraries

Five libraries do all the work:

| Library | Job |
|---|---|
| `transformers` | loads the model and its tokenizer |
| `bitsandbytes` | the 4-bit quantisation that makes **QLoRA** possible |
| `peft` | adds the **LoRA** adapters |
| `trl` | `SFTTrainer` — the supervised fine-tuning loop |
| `datasets` | holds the training examples |

> **Note:** Colab may print a dependency warning and ask you to restart the session. That is normal — restart and
> carry on from the next cell. You do **not** need to re-run this one.

In [ ]:
%pip install -U transformers trl peft datasets accelerate bitsandbytes

print("Installed. If Colab shows a 'Restart session' button, click it, then continue from the NEXT cell.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.0
    Uninstalling transformers-5.15.0:
      Successfully uninstalled transformers-5.15.0
Installed. If Colab shows a 'Restart session' button, click it, then continue from the NEXT cell.


## Step 0b — Check we actually have a GPU

If this prints `GPU: not available`, go to **Runtime → Change runtime type → T4 GPU** and run the cell again.
Everything below will still *run* on CPU, but training would take hours instead of minutes.

In [ ]:
import torch, transformers, trl, peft

print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"trl          {trl.__version__}")
print(f"peft         {peft.__version__}")
print()

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    major = torch.cuda.get_device_capability(0)[0]
    print(f"GPU: {name}  ({total:.1f} GB)")
    is_bf16_supported_cuda = torch.cuda.is_bf16_supported() # Store result
    print(f"torch.cuda.is_bf16_supported(): {is_bf16_supported_cuda}") # New diagnostic

    # Ampere (compute capability 8.0) and newer support bfloat16. A T4 has major=7.
    # Force USE_BF16 to False on T4 GPUs, even if CUDA reports bfloat16 support,
    # as we've encountered issues with PyTorch builds on Colab T4s leading to NotImplementedError.
    if major < 8: # T4 GPU
        USE_BF16 = False
        print("Detected T4 GPU (compute capability < 8), forcing bfloat16 to False.")
    else: # A100 or newer
        USE_BF16 = major >= 8 and is_bf16_supported_cuda
else:
    print("GPU: not available  ->  Runtime > Change runtime type > T4 GPU")
    USE_BF16 = False

DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"Training precision: {'bfloat16' if USE_BF16 else 'float16'}")

torch        2.11.0+cu128
transformers 5.15.1
trl          1.10.0
peft         0.20.0

GPU: Tesla T4  (15.6 GB)
torch.cuda.is_bf16_supported(): True
Detected T4 GPU (compute capability < 8), forcing bfloat16 to False.
Training precision: float16


---
## Step 1 — The use case, and the format we want

**Nimbus** is a fictional cloud file-storage product. Its support team wants every reply to follow the same
four-part shape, so customers can scan it in three seconds:

```
Thanks for reaching out about <the topic>.

What's happening: <one sentence explaining the cause>
What to do: <one concrete action>
If that doesn't help: <the escalation path>

— Nimbus Support
```

A general-purpose model *can* answer these questions — but it will ramble, vary its structure, add
bullet lists, and forget the sign-off. **That is exactly what fine-tuning fixes.** We are not teaching it
new facts; we are teaching it a *behaviour*.

### Why we build the data in code

Real projects start from exported tickets. We generate ours from templates so that:

* the notebook is fully reproducible and needs no download,
* the target format is **100% consistent** — the single most important property of an SFT dataset (Supervised Fine-Tuning),
* you can see exactly what an SFT example looks like.

> **Remember from the deck:** 50–500 *excellent* examples beat 5,000 sloppy ones. We use 130.

In [ ]:
import random, textwrap
random.seed(0)   # reproducible for everyone in the class

SYSTEM = (
    "You are Nimbus Support. Reply to every customer message using exactly this format:\n"
    "Thanks for reaching out about <topic>.\n\n"
    "What's happening: <one sentence>\n"
    "What to do: <one action>\n"
    "If that doesn't help: <escalation>\n\n"
    "— Nimbus Support"
)

# Each issue = the topic label, several ways a customer might phrase it, and the three answer parts.
ISSUES = [
    dict(topic="a password reset email that never arrived",
         asks=["I asked for a password reset 20 minutes ago and nothing has arrived.",
               "No reset email. Can you resend it?",
               "Password reset mail is not coming to my inbox.",
               "I clicked forgot password but never got the mail."],
         cause="Reset emails can take a few minutes and are often filtered into spam or promotions.",
         action="Search your mail for 'Nimbus' including the spam folder, then request a new link from the sign-in page.",
         escalate="Reply here with the email address on the account and we will send a link manually."),
    dict(topic="an upload that keeps failing",
         asks=["My uploads fail at about 80% every time.",
               "Files won't upload, it just stops halfway.",
               "Upload keeps failing with no error message.",
               "I cannot upload anything since this morning."],
         cause="Uploads usually stall when the connection drops or the file is above your plan's single-file limit.",
         action="Retry on a stable network and confirm the file is under 5 GB.",
         escalate="Send us the file name and size and we will check the upload log for your account."),
    dict(topic="being charged twice",
         asks=["I was billed twice this month.",
               "There are two charges on my card for the same subscription.",
               "Double payment taken, please refund one.",
               "My statement shows the Nimbus fee twice."],
         cause="A duplicate charge is normally a pending authorisation that has not yet dropped off the statement.",
         action="Check whether one of the two lines is still marked pending; those clear on their own within five working days.",
         escalate="If both have settled, reply with the last four digits of the card and we will refund one immediately."),
    dict(topic="cancelling the subscription",
         asks=["How do I cancel my plan?",
               "I want to stop my subscription before the next renewal.",
               "Please cancel my account.",
               "Where is the cancel button?"],
         cause="Cancellation is self-serve and takes effect at the end of the current billing period.",
         action="Open Settings, choose Billing, then Cancel plan and confirm.",
         escalate="If the option is greyed out, tell us your workspace name and we will cancel it for you."),
    dict(topic="a shared link that does not work",
         asks=["The share link I sent my client returns an error.",
               "My colleague says the shared folder link is broken.",
               "Shared link says access denied.",
               "People outside my team cannot open the link I shared."],
         cause="Share links default to team-only access and also expire after 30 days.",
         action="Open the file, choose Share, and set the link to 'Anyone with the link' with a new expiry date.",
         escalate="Paste the link here and we will check its permissions from our side."),
    dict(topic="files missing after a sync",
         asks=["Some of my files disappeared after syncing.",
               "The desktop app synced and now a folder is gone.",
               "I lost files following the last sync.",
               "My documents folder is empty after sync."],
         cause="Missing files after a sync are almost always moved rather than deleted, and stay recoverable for 30 days.",
         action="Open the Trash view in the web app and use Restore on the affected folder.",
         escalate="If nothing is in Trash, send the folder name and the approximate time and we will restore from backup."),
    dict(topic="slow download speeds",
         asks=["Downloads are extremely slow today.",
               "It takes forever to download a 200 MB file.",
               "Why is my download speed so bad?",
               "Downloading from Nimbus is much slower than usual."],
         cause="Slow transfers are usually caused by the region setting rather than by your connection.",
         action="Go to Settings, then Storage region, and pick the region closest to you.",
         escalate="If the speed does not improve, send us a screenshot of a speed test and your region setting."),
    dict(topic="enabling two-factor authentication",
         asks=["How do I turn on 2FA?",
               "I want two-factor authentication on my account.",
               "Can I add an authenticator app?",
               "Where do I enable two step login?"],
         cause="Two-factor authentication is available on every plan, including the free one.",
         action="Open Settings, then Security, choose Add authenticator app and scan the QR code.",
         escalate="If the codes are rejected, check your phone's clock is set automatically and then contact us."),
    dict(topic="storage that is full",
         asks=["I've hit my storage limit.",
               "It says my account is full and I cannot upload.",
               "Out of space on Nimbus, what now?",
               "Storage quota exceeded message keeps appearing."],
         cause="Deleted files still count towards your quota while they sit in Trash.",
         action="Empty the Trash from the web app, then check Settings for the largest folders.",
         escalate="If you are still over the limit, reply here and we will show you the upgrade options."),
    dict(topic="adding a teammate to the workspace",
         asks=["How do I invite a colleague?",
               "I need to add someone to my team.",
               "Can I give a new employee access?",
               "Where do I add users to the workspace?"],
         cause="Only workspace owners and admins can send invitations.",
         action="Open Members in the sidebar, click Invite, and enter their work email address.",
         escalate="If you cannot see the Invite button, ask your workspace owner or tell us the workspace name."),
    dict(topic="the mobile app crashing",
         asks=["The Android app crashes on launch.",
               "Nimbus closes immediately on my phone.",
               "The mobile app keeps crashing when I open a folder.",
               "App crashes every time I try to preview a photo."],
         cause="Crashes on open are almost always a stale cache left by an older app version.",
         action="Update the app from the store, then clear its cache from your phone's app settings.",
         escalate="If it still crashes, send us your phone model and app version and we will check the crash log."),
    dict(topic="exporting all of your data",
         asks=["I want to download everything I have stored.",
               "How can I export all my files at once?",
               "Is there a way to get a full backup of my account?",
               "I need a complete export before I leave."],
         cause="Full exports are prepared in the background and delivered as a download link by email.",
         action="Open Settings, then Data, and choose Request full export.",
         escalate="If the email does not arrive within 24 hours, reply here and we will regenerate the link."),
    dict(topic="restoring an older version of a file",
         asks=["Can I go back to yesterday's version of a document?",
               "I overwrote a file, is the old one recoverable?",
               "How does version history work?",
               "I need the previous version of a spreadsheet."],
         cause="Every save creates a version, and versions are kept for 30 days on all paid plans.",
         action="Right-click the file, choose Version history, and restore the timestamp you need.",
         escalate="If the version you need is older than 30 days, send us the file path and we will check backups."),
]

def answer(issue):
    return (
        f"Thanks for reaching out about {issue['topic']}.\n\n"
        f"What's happening: {issue['cause']}\n"
        f"What to do: {issue['action']}\n"
        f"If that doesn't help: {issue['escalate']}\n\n"
        f"— Nimbus Support"
    )

# --- light augmentation -------------------------------------------------------
# Real customers do not all write the same way. We vary the INPUT with realistic
# greetings and sign-offs while keeping the OUTPUT format completely fixed.
# That is exactly the deck's advice: vary the input, hold the output shape constant.
PREFIXES = ["", "Hi, ", "Hello — ", "Hey team, ", "Quick one: ", "Morning, "]
SUFFIXES = ["", " Thanks!", " Please advise.", " Any ideas?", " This is urgent.", " Appreciate the help."]

VARIANTS_PER_ISSUE = 10

rows = []
for issue in ISSUES:
    seen = set()
    target = answer(issue)
    while len(seen) < VARIANTS_PER_ISSUE:
        ask = random.choice(issue["asks"])
        text = random.choice(PREFIXES) + ask + random.choice(SUFFIXES)
        if text in seen:
            continue
        seen.add(text)
        rows.append({"messages": [
            {"role": "system",    "content": SYSTEM},
            {"role": "user",      "content": text},
            {"role": "assistant", "content": target},
        ]})

random.shuffle(rows)
print(f"Issue types : {len(ISSUES)}")
print(f"Total examples built: {len(rows)}")
print(f"\nA sample of how varied the customer messages are:")
for r in rows[:5]:
    print("  -", r["messages"][1]["content"])

Issue types : 13
Total examples built: 130

A sample of how varied the customer messages are:
  - Quick one: The mobile app keeps crashing when I open a folder.
  - Quick one: It takes forever to download a 200 MB file. Appreciate the help.
  - Morning, The desktop app synced and now a folder is gone. Appreciate the help.
  - Hey team, I've hit my storage limit. Thanks!
  - The mobile app keeps crashing when I open a folder. Thanks!


### Split the data — **before** we train, not after

We hold out 15 examples the model will **never** see. Without a held-out set you cannot tell improvement
from luck, and "it looks better" is not a result you can report to anyone.

> **Be honest about what your eval measures.** Here the same *issue types* appear in both splits, worded
> differently. So this tests **format consistency on unseen phrasings** — not generalisation to issues the
> model has never encountered. In a real project you would also hold out whole categories. Knowing which
> question your evaluation answers is half of evaluation.

In [ ]:
from datasets import Dataset

eval_rows  = rows[:15]
train_rows = rows[15:]

train_ds = Dataset.from_list(train_rows)
eval_ds  = Dataset.from_list(eval_rows)

print(f"train: {len(train_ds)} examples")
print(f"held-out eval: {len(eval_ds)} examples  (the model never sees these)")

# What one training example actually looks like:
ex = train_ds[0]
print("\n" + "=" * 78)
for m in ex["messages"]:
    print(f"\n[{m['role'].upper()}]\n{m['content']}")
print("\n" + "=" * 78)

train: 115 examples
held-out eval: 15 examples  (the model never sees these)


[SYSTEM]
You are Nimbus Support. Reply to every customer message using exactly this format:
Thanks for reaching out about <topic>.

What's happening: <one sentence>
What to do: <one action>
If that doesn't help: <escalation>

— Nimbus Support

[USER]
Quick one: My uploads fail at about 80% every time.

[ASSISTANT]
Thanks for reaching out about an upload that keeps failing.

What's happening: Uploads usually stall when the connection drops or the file is above your plan's single-file limit.
What to do: Retry on a stable network and confirm the file is under 5 GB.
If that doesn't help: Send us the file name and size and we will check the upload log for your account.

— Nimbus Support



### What the model actually sees

The list of `messages` above is *not* what reaches the model. The tokenizer's **chat template** turns it into
one string with the special tokens this model family was trained with.

Never hand-write this string yourself — every model family uses different markers, and getting it subtly wrong
is one of the hardest fine-tuning bugs to diagnose. Let the tokenizer do it.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # Apache-2.0, ungated, ~0.5B parameters

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

formatted = tokenizer.apply_chat_template(train_ds[0]["messages"], tokenize=False)
print(formatted)
print("\n--- token count for this example:", len(tokenizer(formatted)["input_ids"]))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

<|im_start|>system
You are Nimbus Support. Reply to every customer message using exactly this format:
Thanks for reaching out about <topic>.

What's happening: <one sentence>
What to do: <one action>
If that doesn't help: <escalation>

— Nimbus Support<|im_end|>
<|im_start|>user
Quick one: My uploads fail at about 80% every time.<|im_end|>
<|im_start|>assistant
Thanks for reaching out about an upload that keeps failing.

What's happening: Uploads usually stall when the connection drops or the file is above your plan's single-file limit.
What to do: Retry on a stable network and confirm the file is under 5 GB.
If that doesn't help: Send us the file name and size and we will check the upload log for your account.

— Nimbus Support<|im_end|>


--- token count for this example: 163


---
## Step 2 — Load the model in 4-bit (this is the "Q" in QLoRA)

`BitsAndBytesConfig` is the whole QLoRA memory trick in one object:

| Setting | What it does |
|---|---|
| `load_in_4bit=True` | store the frozen weights in 4 bits instead of 16 |
| `bnb_4bit_quant_type="nf4"` | **NormalFloat4** — places its 16 levels where neural-network weights actually cluster |
| `bnb_4bit_use_double_quant=True` | quantise the quantisation constants too — a small free saving |
| `bnb_4bit_compute_dtype` | maths still happens in 16-bit, so accuracy is preserved |

The **LoRA adapters we add next are not quantised** — they train in 16-bit. Compress the frozen knowledge;
keep the part that is learning precise.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, #quantization constants themselves are also quantized
    bnb_4bit_compute_dtype=DTYPE,   # weights are stored in 4-bit, the actual computations (like matrix multiplications)
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None,
    dtype=DTYPE,
)
model.config.use_cache = False          # required when gradient checkpointing is on

# Removed explicit .half() call as it's not supported for quantized models.

if torch.cuda.is_available():
    print(f"Model loaded. GPU memory in use: {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("Model loaded on CPU (training will be slow).")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. GPU memory in use: 0.46 GB


---
## Step 3 — The baseline: what does it say **before** training?

**Do not skip this cell.** Recording the "before" is the difference between an experiment and a guess.

Watch for: the model knows plenty about password resets and uploads — it is *not* ignorant. It simply
answers in a shape nobody asked for. Length varies, the four labels are missing, the sign-off is absent.

In [ ]:
from transformers import GenerationConfig

def generate(model, user_message, system=SYSTEM, max_new_tokens=180):
    """Format one message with the chat template and return the model's reply."""
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user_message}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                 # greedy = repeatable, good for a demo. picks the token with the highest probability at each step
            pad_token_id=tokenizer.pad_token_id, #to ensure that all sequences in a batch have the same length.
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


# Three held-out questions we will re-use after training.
demo_questions = [eval_rows[i]["messages"][1]["content"] for i in range(3)]

before = {}
model.eval()
for q in demo_questions:
    before[q] = generate(model, q)
    print("=" * 78)
    print(f"QUESTION: {q}\n")
    print(before[q])
    print()

QUESTION: Quick one: The mobile app keeps crashing when I open a folder.

Thanks for reaching out about the mobile app crashing when opening folders. We're working on a fix right away. Please let us know if you need any assistance with it.

QUESTION: Quick one: It takes forever to download a 200 MB file. Appreciate the help.

Thanks for reaching out about your file downloading issue. We're here to assist you.

What's happening: The file is taking too long to download, which could be due to various reasons such as network congestion or server issues.

What to do: Please provide more details on what happened and how we can resolve it faster. This will help us understand the situation better and find solutions together.

If that doesn't help: Escalate by providing additional information or screenshots if possible.

QUESTION: Morning, The desktop app synced and now a folder is gone. Appreciate the help.

Thanks for reaching out about your issue with the desktop app syncing. We're here to a

### Score the baseline with a rule, not a feeling

Our target format is mechanical, so we can check it mechanically. Four markers must appear, in order.
This is a tiny **evaluation harness** — the deck's "minimum honest evaluation", in eight lines of Python.

In [ ]:
MARKERS = ["Thanks for reaching out about", "What's happening:", "What to do:", "If that doesn't help:", "— Nimbus Support"]

def format_score(text):
    """Fraction of the required format markers present (0.0 to 1.0)."""
    return sum(m in text for m in MARKERS) / len(MARKERS)

def evaluate(model, rows, label):
    scores = []
    for r in rows:
        reply = generate(model, r["messages"][1]["content"])
        scores.append(format_score(reply))
    avg = sum(scores) / len(scores)
    print(f"{label:<28} format score: {avg:.2f}   ({sum(s == 1.0 for s in scores)}/{len(scores)} perfect)")
    return avg

baseline_score = evaluate(model, eval_rows, "BEFORE fine-tuning")

BEFORE fine-tuning           format score: 0.48   (0/15 perfect)


---
## Step 4 — Attach the LoRA adapters (the "LoRA" in QLoRA)

Recall the formula from the deck:

$$W' = W + \frac{\alpha}{r} \cdot B A$$

`W` stays frozen. We only train the thin `A` and `B` matrices.
Through backpropagation, the model learns the optimal values for the elements within A and B to minimize the loss function (e.g., cross-entropy loss for language generation tasks)
| Argument | Value | Why |
|---|---|---|
| `r` | `16` | rank = adapter capacity. 8–16 is right for a style/format task |
| `lora_alpha` | `32` | scaling; the `2 × r` convention keeps the effective strength constant |
| `lora_dropout` | `0.05` | light regularisation on a small dataset |
| `target_modules` | attention + MLP | the QLoRA paper found *which* layers get adapters matters more than the rank |
| `task_type` | `CAUSAL_LM` | we are training a text generator |

**Watch the printed percentage.** That number — under 2% of the model — is the entire reason this runs on a free GPU.
(With only the four attention projections it drops to about 0.44%; adding the MLP layers buys capacity.)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if torch.cuda.is_available():
    # casts layer norms to fp32 and enables gradient checkpointing: standard QLoRA preparation
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [ ]:
print(f"Model dtype: {model.dtype}")
if hasattr(model, 'hf_device_map'):
    print(f"Model device map: {model.hf_device_map}")
else:
    print(f"Model device: {model.device}")

Model dtype: torch.float32
Model device: cuda:0


---
## Step 5 — Train with `SFTTrainer`

`SFTTrainer` handles the chat template, tokenisation, batching and the training loop. You supply the data
and a handful of settings.

| Argument | Value | Why |
|---|---|---|
| `learning_rate` | `2e-4` | LoRA wants 10–100× more than full fine-tuning — small, freshly-initialised matrices need a bigger push |
| `num_train_epochs` | `3` | small dataset; more than this and it starts memorising |
| `per_device_train_batch_size` × `gradient_accumulation_steps` | `2 × 4 = 8` | simulate a batch of 8 on a GPU that only fits 2 |
| `max_length` | `768` | just above our longest example — longer costs quadratically more memory |



The target_modules parameter in LoraConfig is used to specify which layers of the base model the LoRA adapters will be attached to. Instead of fine-tuning all the parameters of the large pre-trained model, LoRA introduces small, trainable adapter matrices (the A and B matrices in the formula mentioned in the notebook) into specific layers.

When target_modules is set to ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], it means you are applying LoRA adapters to:

    Attention Layers (q_proj, k_proj, v_proj, o_proj): These are the query, key, value, and output projection matrices within the self-attention mechanism of the transformer model. Modifying these allows the model to learn new ways of computing attention, which is crucial for understanding relationships between tokens.
    MLP (Multi-Layer Perceptron) Layers (gate_proj, up_proj, down_proj): These are the feed-forward network components within each transformer block. They are responsible for processing the information further after the attention mechanism. Targeting these allows the model to learn new non-linear transformations and feature representations.

The QLoRA paper (and common practice) found that applying adapters to all linear layers within the attention and MLP blocks is highly effective. This approach allows the model to adapt its core computational mechanisms to the new task (like generating support tickets in a specific format) while keeping the majority of the pre-trained weights frozen, which significantly reduces memory usage and training time.

> **About the compatibility block below:** TRL's API moves fast (`max_seq_length` was renamed `max_length`,
> `tokenizer` became `processing_class`). Rather than pin you to one version, we filter our settings against
> whatever version you installed. This is a genuinely useful pattern for any fast-moving library.

In [ ]:
import inspect
import os
from trl import SFTTrainer, SFTConfig

def accepts(cls) -> set:
    """Names of the keyword arguments this installed version of `cls` accepts."""
    return set(inspect.signature(cls.__init__).parameters)

def keep_supported(cls, wanted: dict) -> dict:
    allowed = accepts(cls)
    return {k: v for k, v in wanted.items() if k in allowed}

def first_supported(cls, candidates: dict) -> dict:
    """For arguments that were renamed: use the first candidate this version accepts."""
    allowed = accepts(cls)
    for name, value in candidates.items():
        if name in allowed:
            return {name: value}
    return {}

# --- Set environment variables to explicitly force float16 for accelerate/bitsandbytes ---
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"
os.environ["BITSANDBYTES_COMPUTE_DTYPE"] = "float16"

# --- training settings --------------------------------------------------------
cfg = dict(
    output_dir="nimbus-support-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="no",
    report_to=[],
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    # Explicitly disabling mixed precision (fp16 and bf16) to remove GradScaler and bypass BFloat16 error.
    fp16=False,
    bf16=False,
    torch_dtype=DTYPE, # Still passing this for consistency, though AMP is off
    gradient_checkpointing=torch.cuda.is_available(),
    gradient_checkpointing_kwargs={"use_reentrant": False},   # required with PEFT
)
# Arguments that were renamed between versions — take whichever this install has.
cfg.update(first_supported(SFTConfig, {"max_length": 768, "max_seq_length": 768}))
cfg.update(first_supported(SFTConfig, {"warmup_ratio": 0.05, "warmup_steps": 5}))

sft_config = SFTConfig(**keep_supported(SFTConfig, cfg))

# --- trainer ------------------------------------------------------------------
tr = dict(model=model, args=sft_config, train_dataset=train_ds)
# `tokenizer` was renamed `processing_class` — again, take whichever exists.
tr.update(first_supported(SFTTrainer, {"processing_class": tokenizer, "tokenizer": tokenizer}))

trainer = SFTTrainer(**keep_supported(SFTTrainer, tr))

print("Starting training — roughly 5 to 10 minutes on a free T4.\n")
trainer.train()
print("\nDone.")

Tokenizing train dataset:   0%|          | 0/115 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/115 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/115 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/115 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training — roughly 5 to 10 minutes on a free T4.



NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

### Reading the training log

* **`loss` should fall** over the run. A flat loss usually means the learning rate is too low.
* A loss that drops to almost zero on a small dataset means the model is **memorising**, not learning.
  On a real project you would watch *evaluation* loss and stop when it turns upward.

---
## Step 6 — The same questions, after training

This is the moment the lab is built around. Same model, same held-out questions, same greedy decoding —
only the adapter is new.

In [ ]:
model.eval()
model.config.use_cache = True
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

for q in demo_questions:
    after = generate(model, q)
    print("=" * 78)
    print(f"QUESTION: {q}\n")
    print("--- BEFORE ---------------------------------------------------------")
    print(textwrap.fill(before[q], 100)[:900])
    print("\n--- AFTER ----------------------------------------------------------")
    print(after)
    print()

In [ ]:
tuned_score = evaluate(model, eval_rows, "AFTER fine-tuning")

print()
print(f"Baseline : {baseline_score:.2f}")
print(f"Tuned    : {tuned_score:.2f}")
print(f"Change   : {tuned_score - baseline_score:+.2f}")
print("\n(Format compliance on 15 examples the model never saw during training.)")

---
## Step 7 — Save the adapter

The adapter is a **file you own**. Compare its size with the base model it modifies — this is the
"one base model, many adapters" economics from the deck, made concrete.

In [ ]:
import os

SAVE_DIR = "nimbus-support-lora/adapter"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

total = 0
print(f"Contents of {SAVE_DIR}:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    total += size
    print(f"  {f:<32} {size/1e6:8.2f} MB")

print(f"\nAdapter total: {total/1e6:.1f} MB")
print(f"Base model   : ~{0.5 * 2:.0f} GB in 16-bit  (Qwen2.5-0.5B)")
print("\nA 7B base model would be ~14 GB — and the adapter would still be only a few MB.")

### How you would load it again later

You do **not** ship the whole model — you ship the base model reference plus this small folder.

```python
from transformers import AutoModelForCausalLM
from peft import PeftModel

base  = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")
tuned = PeftModel.from_pretrained(base, "nimbus-support-lora/adapter")

# Optional: fold the adapter into the weights for zero inference overhead.
# Only do this on a 16-bit base — not on a 4-bit quantised one.
# tuned = tuned.merge_and_unload()
```

To download the adapter out of Colab:

```python
from google.colab import files
!zip -r adapter.zip nimbus-support-lora/adapter
files.download("adapter.zip")
```

---
## Step 8 — **TRY EDITING THIS** (the part that teaches the most)

Pick one. Each takes a few minutes and each demonstrates a real lesson from the deck.

| # | Change | What you should observe | Concept |
|---|---|---|---|
| 1 | Change the format in `answer()` — e.g. add a `Reference:` line or drop the sign-off, then re-run everything from the dataset cell | The model learns the *new* shape just as reliably. **You control the behaviour through the data.** | Data is the curriculum |
| 2 | Set `num_train_epochs=10` | Training loss drops near zero, but held-out replies get rigid and start reusing the wrong issue's text | Overfitting |
| 3 | Set `r=4` and `lora_alpha=8` | Often still works. Capacity is rarely the bottleneck for a format task | Rank is not a magic knob |
| 4 | Cut the dataset to 20 examples (`train_ds = Dataset.from_list(train_rows[:20])`) | The score drops sharply — the clearest demonstration of why data volume matters | Data quantity |
| 5 | Ask an **off-topic** question, e.g. `generate(model, "What is the capital of France?")` | It may now answer in support format anyway | Catastrophic forgetting |
| 6 | Swap `MODEL_ID` to `"Qwen/Qwen3-0.6B"` or `"HuggingFaceTB/SmolLM2-360M-Instruct"` | Same code, different base model | Portability of the recipe |

```python
# Exercise 5 — run this cell as-is to see catastrophic forgetting in action
print(generate(model, "What is the capital of France?"))
```

In [ ]:
# Exercise 5, ready to run:
print(generate(model, "What is the capital of France?"))

---
## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `CUDA out of memory` | batch or sequence length too large | `Runtime → Restart session`, then set `per_device_train_batch_size=1` and `gradient_accumulation_steps=8` |
| `No GPU` / training crawls | CPU runtime | `Runtime → Change runtime type → T4 GPU` |
| `ImportError` after installing | Colab needs a restart | Restart the session, then re-run from the *env check* cell (skip the install) |
| `TypeError: __init__() got an unexpected keyword argument ...` | TRL version drift | The `supported()` filter handles the known ones; for anything else, check the docs for *your* installed version |
| Loss stays flat | learning rate too low, or adapters not attached | Confirm `print_trainable_parameters()` printed a non-zero count; try `learning_rate=3e-4` |
| Replies repeat one issue's text | over-training or too little variety | Fewer epochs, or add more distinct issues to `ISSUES` |
| `bitsandbytes` errors on a non-NVIDIA machine | 4-bit needs CUDA | Remove `quantization_config` and train in fp32 on CPU (slow) — or use Colab |

---

## What you just did — mapped back to the deck

| Notebook step | Deck slide |
|---|---|
| Built a consistent dataset, held out 15 rows | Slide 9 — *Data quality and evaluation decide the outcome* |
| Loaded the model in 4-bit NF4 | Slide 7 — *QLoRA* |
| Attached LoRA, saw `<1%` trainable | Slides 4–5 — *PEFT / LoRA* |
| Trained with `SFTTrainer` | Slide 8 — *SFT* |
| Scored format compliance before vs after | Slide 9 — *Minimum honest evaluation* |
| Saved a few-MB adapter you own | Slides 12–13 — *Open weights are the hedge* |

**Not covered here:** GRPO. It needs many generations per prompt and far more compute than free Colab
gives. See slide 10 for the theory and TRL's `GRPOTrainer` docs when you have a GPU budget.

---

### Reference links

* Hugging Face PEFT — <https://huggingface.co/docs/peft>
* TRL (SFT, DPO, GRPO trainers) — <https://huggingface.co/docs/trl>
* Unsloth free notebooks — <https://github.com/unslothai/unsloth>
* LoRA paper — arXiv 2106.09685 · QLoRA paper — arXiv 2305.14314